<a href="https://colab.research.google.com/github/goitstudent123/llm-gen/blob/main/dz_topic4_has.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
# !pip install openai
import os, json
from openai import OpenAI
import json
import re

# 1) Ключ краще зберігати у змінній середовища
client = OpenAI(base_url="https://openrouter.ai/api/v1",
                api_key=os.getenv("OPENROUTER_API_KEY"))
MODEL = "openai/gpt-oss-120b:nitro"  # замінюйте за потреби


In [55]:
def normalize_text(value: str) -> str:
    value = value.lower()
    value = value.replace("-", "-").replace("–", "-").replace("—", "-")
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def model_semantic_match(expected: str, actual: str) -> bool:
    prompt = """
You compare two short texts.

Return ONLY valid JSON:
{"match": true} or {"match": false}

Rules:
- true: same meaning, same key facts, minor wording/punctuation/spacing differences are OK.
- true: synonyms like "та" and "і" are OK.
- false: different numbers, versions, dates, entities, actions, or meaning.
- false: actual misses an important fact from expected.
"""

    user_message = f"""
Expected:
<<<{expected}>>>

Actual:
<<<{actual}>>>
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": prompt.strip()},
            {"role": "user", "content": user_message.strip()},
        ],
        temperature=0,
    )

    raw_result = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(raw_result)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model returned invalid JSON: {raw_result}") from e

    return parsed["match"] is True


def smart_string_match(expected: str, actual: str) -> bool:
    if expected == actual:
        return True

    if normalize_text(expected) == normalize_text(actual):
        return True

    return model_semantic_match(expected, actual)

In [56]:
def execute(prompt: str, texts: list[str]) -> list[dict]:
    results = []

    for text in texts:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "developer",
                    "content": prompt.strip(),
                },
                {
                    "role": "user",
                    "content": text,
                },
            ],
            temperature=0,
        )

        raw_result = resp.choices[0].message.content.strip()

        try:
            parsed_result = json.loads(raw_result)
        except json.JSONDecodeError as e:
            raise ValueError(f"Model returned invalid JSON: {raw_result}") from e

        results.append(parsed_result)
        print(parsed_result)

    return results

def compare(actual_results: list[dict], expected_results: list[dict]) -> None:
    max_len = max(len(actual_results), len(expected_results))
    has_differences = False

    for index in range(max_len):
        line_number = index + 1

        if index >= len(expected_results):
            has_differences = True
            print(f"Line {line_number}: unexpected extra actual result: {actual_results[index]}")
            continue

        if index >= len(actual_results):
            has_differences = True
            print(f"Line {line_number}: missing actual result, expected: {expected_results[index]}")
            continue

        actual = actual_results[index]
        expected = expected_results[index]

        line_diffs = []

        all_keys = sorted(set(actual.keys()) | set(expected.keys()))

        for key in all_keys:
            actual_value = actual.get(key)
            expected_value = expected.get(key)

            if not smart_string_match(actual_value,expected_value):
                line_diffs.append(
                    f"{key}: expected={expected_value!r}, actual={actual_value!r}"
                )

        if line_diffs:
            has_differences = True
            print(f"Line {line_number}:")
            for diff in line_diffs:
                print(f"  - {diff}")

    if not has_differences:
        print("All results match.")

In [57]:
student_texts = [
    "Оксана Дрозд натхненно провела демо — команда аплодувала стоячи.",
    "Керівник відділу Андрій Король грубо порушив інструкції, через що реліз зірвано.",
    "Роман Бондаренко надіслав звіт о 14:00 і підтвердив графік зустрічей.",
    "Сьогодні Ілля Ткачук блискуче оптимізував код і заощадив 30% бюджету.",
    "Марко Савченко ігнорує повідомлення й затримує погодження — це неприйнятно."
]

PROMPT_STUDENT = """
Extract from a Ukrainian sentence. Output ONLY JSON: {"person":"<full name as written>","tone":"позитивна|нейтральна|негативна"}

person: name exactly as in text, no titles.
tone:
- позитивна: praise, success
- негативна: blame, failure, unacceptable
- нейтральна: factual, no evaluation

Input: <<<{{TEXT}}>>>
Output:
"""

expected_results = [
    {"person": "Оксана Дрозд", "tone": "позитивна"},
    {"person": "Андрій Король", "tone": "негативна"},
    {"person": "Роман Бондаренко", "tone": "нейтральна"},
    {"person": "Ілля Ткачук", "tone": "позитивна"},
    {"person": "Марко Савченко", "tone": "негативна"},
]

actual_results = execute(PROMPT_STUDENT, student_texts)

compare(actual_results, expected_results)


{'person': 'Оксана Дрозд', 'tone': 'позитивна'}
{'person': 'Андрій Король', 'tone': 'негативна'}
{'person': 'Роман Бондаренко', 'tone': 'нейтральна'}
{'person': 'Ілля Ткачук', 'tone': 'позитивна'}
{'person': 'Марко Савченко', 'tone': 'негативна'}
All results match.


In [58]:

product_texts = [
"Квитанції не завантажуються в Atlas Pro з мобільного — фінвідділ стоїть",
"Після оновлення 3.1 у Core графіки будуються на 2 секунди довше, але працюють стабільно",
"Автовідповідь у Desk на свята — nice-to-have, допоможе трохи зменшити навантаження на операторів",
"Core падає під час імпорту CSV >100 тис. рядків — роботу заблоковано",
"Atlas показує старі курси валют, поки що оновлюємо вручну"
]

PROMPT_PRODUCT = """
Classify a n ticket. Output ONLY JSON: {"product":"Atlas|Core|Desk","priority":"високий|середній|низький"}

product: name mentioned in text (ignore Pro/Lite/versions). If multiple, pick the one the issue is about.

priority (by impact, not wording):
- високий: blocked work, crash, data loss/incorrectness, no workaround, financial risk
- середній: degraded but working; manual workaround exists; slower UX
- низький: nice-to-have, cosmetic, minor optimization

Data-integrity/financial issues = високий even with a manual workaround.

Input: <<<{{TEXT}}>>>
Output:
"""

expected_results = [
        {"product":"Atlas","priority":"високий"},
        {"product":"Core","priority":"середній"},
        {"product":"Desk","priority":"низький"},
        {"product":"Core","priority":"високий"},
        {"product":"Atlas","priority":"високий"}
]

actual_results = execute(PROMPT_PRODUCT, product_texts)

compare(actual_results, expected_results)

{'product': 'Atlas', 'priority': 'високий'}
{'product': 'Core', 'priority': 'середній'}
{'product': 'Desk', 'priority': 'низький'}
{'product': 'Core', 'priority': 'високий'}
{'product': 'Atlas', 'priority': 'високий'}
All results match.


In [59]:

criterias_texts = [
"Вийшла версія 2.0 з офлайн-режимом і новим дизайном",
"У частини користувачів не відкривається профіль після апдейту; працюємо над виправленням",
"Збираємо відгуки про новий модуль аналітики",
"На вихідних сервіс у режимі планового обслуговування",
"Рекомендуємо вмикати двофакторну автентифікаці"
]

PROMPT_CRITERIAS = """
Summarize a announcement. Output ONLY JSON: {"title":"<≤8 words, Ukrainian>","category":"реліз|інцидент|рекомендація"}

title: ≤8 words, no "Оголошення:" prefix, no quotes, capture the essence.
category:
- реліз: new version, feature launch
- інцидент: bug, outage, maintenance, service disruption
- рекомендація: advice, feedback request, suggestion

Input: <<<{{TEXT}}>>>
Output:
"""

expected_results = [
        {"title":"Версія 2.0 з офлайн‑режимом та новим дизайном","category":"реліз"},
        {"title":"Проблема з відкриттям профілю після оновлення","category":"інцидент"},
        {"title":"Збір відгуків про новий модуль аналітики","category":"рекомендація"},
        {"title":"Сервіс у режимі планового обслуговування","category":"інцидент"},
        {"title":"Вмикайте двофакторну автентифікацію","category":"рекомендація"}
]

actual_results = execute(PROMPT_CRITERIAS, criterias_texts)

compare(actual_results, expected_results)

{'title': 'Версія 2.0: офлайн‑режим і новий дизайн', 'category': 'реліз'}
{'title': 'Проблема з відкриттям профілю після оновлення', 'category': 'інцидент'}
{'title': 'Збір відгуків про новий модуль аналітики', 'category': 'рекомендація'}
{'title': 'Сервіс у режимі планового обслуговування', 'category': 'інцидент'}
{'title': 'Вмикайте двофакторну автентифікацію', 'category': 'рекомендація'}
All results match.
